# Level 2 — Task 5: Autocomplete and Autocorrect Data Analytics

## Objective
Analyse a text corpus and build simple data-driven **autocomplete** and **autocorrect** components.

The project focuses on:
- text cleaning and tokenization
- word-frequency analysis
- bigram and trigram patterns
- next-word autocomplete
- edit-distance based autocorrect
- practical NLP insights

## Dataset
The notebook uses the public **NLTK Gutenberg corpus**, which contains books and literary texts. It downloads the corpus automatically.

This is appropriate for learning language patterns because it provides a large collection of naturally occurring English sentences.

## Workflow
1. Download and inspect the corpus.
2. Clean and tokenize text.
3. Analyse word frequencies.
4. Analyse common bigrams/trigrams.
5. Build a prefix-based autocomplete function.
6. Build a context-aware next-word suggestion function.
7. Build an edit-distance autocorrect function.
8. Test the system on example inputs.
9. Save vocabulary and frequency outputs.
10. Discuss limitations and real-world improvements.

**Tools:** Python, Pandas, NumPy, NLTK, Matplotlib, Seaborn.


In [ ]:
import os
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from collections import Counter, defaultdict

nltk.download("gutenberg", quiet=True)
nltk.download("punkt", quiet=True)

from nltk.corpus import gutenberg
from nltk.tokenize import sent_tokenize, word_tokenize

print("Available Gutenberg files:", len(gutenberg.fileids()))
print(gutenberg.fileids()[:10])


## 1. Load and inspect the corpus


In [ ]:
# Use several Gutenberg books to create a reasonably broad English corpus
selected_files = gutenberg.fileids()

raw_text = "\n".join(gutenberg.raw(fid) for fid in selected_files)

print("Characters:", len(raw_text))
print("Books:", len(selected_files))
print("First 500 characters:")
print(raw_text[:500])


## 2. Text cleaning and tokenization

The preprocessing keeps alphabetic words and converts everything to lowercase. This reduces noise caused by punctuation and capitalization.


In [ ]:
tokens = re.findall(r"[a-z]+", raw_text.lower())

print("Total word tokens:", len(tokens))
print("Unique words:", len(set(tokens)))
print("First 30 tokens:", tokens[:30])


In [ ]:
word_counts = Counter(tokens)

top_words = pd.DataFrame(
    word_counts.most_common(20),
    columns=["Word", "Frequency"]
)

display(top_words)

plt.figure(figsize=(10, 7))
sns.barplot(data=top_words, x="Frequency", y="Word")
plt.title("Top 20 Most Frequent Words")
plt.show()


**Observation:** Natural-language corpora have a highly uneven frequency distribution. A small group of common words appears very often, while a long tail of words occurs much less frequently.


## 3. Bigram and trigram analysis

Bigrams are two-word sequences and trigrams are three-word sequences. They capture local language patterns and are useful for simple autocomplete systems.


In [ ]:
bigrams = Counter(zip(tokens[:-1], tokens[1:]))
trigrams = Counter(zip(tokens[:-2], tokens[1:-1], tokens[2:]))

top_bigrams = pd.DataFrame(
    [(" ".join(k), v) for k, v in bigrams.most_common(20)],
    columns=["Bigram", "Frequency"]
)

top_trigrams = pd.DataFrame(
    [(" ".join(k), v) for k, v in trigrams.most_common(20)],
    columns=["Trigram", "Frequency"]
)

print("Top bigrams")
display(top_bigrams.head(10))

print("Top trigrams")
display(top_trigrams.head(10))


In [ ]:
plt.figure(figsize=(10, 7))
sns.barplot(data=top_bigrams.head(15), x="Frequency", y="Bigram")
plt.title("Top 15 Bigrams")
plt.show()


## 4. Prefix-based autocomplete

A basic autocomplete system can suggest common words beginning with a user's partially typed prefix.


In [ ]:
# Keep words that occur at least twice to reduce extremely rare suggestions
vocabulary = {w: c for w, c in word_counts.items() if len(w) >= 2 and c >= 2}

def autocomplete(prefix, n=5):
    prefix = prefix.lower().strip()
    matches = [(word, count) for word, count in vocabulary.items()
               if word.startswith(prefix)]
    matches.sort(key=lambda x: (-x[1], x[0]))
    return pd.DataFrame(matches[:n], columns=["Suggestion", "Frequency"])

for prefix in ["th", "comp", "data", "auto"]:
    print("Prefix:", prefix)
    display(autocomplete(prefix))


**Observation:** Prefix autocomplete is simple and fast because it only needs to search words that start with the entered characters. A production system would normally use a Trie or a compressed search index for much larger vocabularies.


## 5. Context-aware next-word autocomplete

A more useful system uses the previous word as context. Here, the most frequent words following a given word are suggested.


In [ ]:
next_words = defaultdict(Counter)

for w1, w2 in bigrams:
    next_words[w1][w2] += bigrams[(w1, w2)]

def next_word_suggestions(previous_word, n=5):
    previous_word = previous_word.lower().strip()
    suggestions = next_words.get(previous_word, Counter()).most_common(n)
    return pd.DataFrame(suggestions, columns=["Next Word", "Frequency"])

for word in ["the", "data", "she", "he"]:
    print("Previous word:", word)
    display(next_word_suggestions(word))


## 6. Edit-distance autocorrect

Autocomplete predicts what a user may want to type next. Autocorrect instead tries to find a likely intended word when the typed word contains a spelling error.

This notebook implements Levenshtein edit distance:
- insertion = 1
- deletion = 1
- substitution = 1

The candidate word with the smallest edit distance is preferred, with frequency used as a tie-breaker.


In [ ]:
def levenshtein_distance(a, b):
    if a == b:
        return 0

    if len(a) < len(b):
        a, b = b, a

    previous = list(range(len(b) + 1))

    for i, char_a in enumerate(a, start=1):
        current = [i]
        for j, char_b in enumerate(b, start=1):
            insert_cost = current[j - 1] + 1
            delete_cost = previous[j] + 1
            replace_cost = previous[j - 1] + (char_a != char_b)
            current.append(min(insert_cost, delete_cost, replace_cost))
        previous = current

    return previous[-1]

def autocorrect(word, n=5, max_distance=2):
    word = word.lower().strip()

    candidates = []
    for candidate, frequency in vocabulary.items():
        distance = levenshtein_distance(word, candidate)
        if distance <= max_distance:
            candidates.append((candidate, distance, frequency))

    candidates.sort(key=lambda x: (x[1], -x[2], x[0]))

    return pd.DataFrame(
        candidates[:n],
        columns=["Suggestion", "Edit Distance", "Frequency"]
    )

for typo in ["recieve", "teh", "becuse", "langauge", "analysiss"]:
    print("Typed word:", typo)
    display(autocorrect(typo))


## 7. Evaluate simple examples

The examples below demonstrate how the autocomplete and autocorrect components behave on unseen user input.


In [ ]:
examples = ["recieve", "becuse", "langauge", "analysiss", "computr"]

example_rows = []
for word in examples:
    result = autocorrect(word, n=1)
    suggestion = result.iloc[0]["Suggestion"] if len(result) else None
    distance = result.iloc[0]["Edit Distance"] if len(result) else None
    example_rows.append({
        "Typed Word": word,
        "Suggested Correction": suggestion,
        "Edit Distance": distance
    })

example_results = pd.DataFrame(example_rows)
display(example_results)


## 8. Frequency vs vocabulary coverage

A useful NLP insight is the trade-off between vocabulary size and coverage. Keeping every rare word creates a huge search space, while keeping only frequent words improves speed but may remove valid words.


In [ ]:
frequency_thresholds = [1, 2, 3, 5, 10, 20, 50, 100]
vocab_sizes = [sum(c >= t for c in word_counts.values()) for t in frequency_thresholds]

coverage_df = pd.DataFrame({
    "Minimum Frequency": frequency_thresholds,
    "Vocabulary Size": vocab_sizes
})

display(coverage_df)

plt.figure(figsize=(9, 5))
sns.lineplot(data=coverage_df, x="Minimum Frequency", y="Vocabulary Size", marker="o")
plt.title("Vocabulary Size vs Minimum Frequency")
plt.ylabel("Number of Words")
plt.show()


## 9. Practical recommendations

### Autocomplete
- Use prefix frequency for fast basic suggestions.
- Use previous-word context for more relevant next-word predictions.
- A production application can use a Trie for prefix lookup and n-gram/Transformer models for context.

### Autocorrect
- Use edit distance to generate candidate spellings.
- Combine edit distance with word frequency.
- For high-quality correction, use sentence context because the closest spelling is not always the intended word.

### Limitations
- Gutenberg is literary English and may not represent modern chat, social media or Indian-English usage.
- Simple bigrams cannot understand long-range context.
- Edit distance does not understand meaning.
- A production system should use a much larger and domain-specific corpus.


In [ ]:
# Save outputs
top_words.to_csv("../output/top_words.csv", index=False)
top_bigrams.to_csv("../output/top_bigrams.csv", index=False)
top_trigrams.to_csv("../output/top_trigrams.csv", index=False)
coverage_df.to_csv("../output/vocabulary_frequency_analysis.csv", index=False)
example_results.to_csv("../output/autocorrect_examples.csv", index=False)

# Save a compact vocabulary file for inspection
vocab_output = pd.DataFrame(
    sorted(vocabulary.items(), key=lambda x: -x[1]),
    columns=["Word", "Frequency"]
)
vocab_output.head(5000).to_csv("../output/vocabulary_top_5000.csv", index=False)

print("Autocomplete and Autocorrect Data Analytics task completed.")


## Conclusion

This project demonstrates how basic statistical language modelling can support autocomplete and autocorrect.

Word frequencies provide the foundation, bigrams provide short context, and edit distance provides a practical spelling-correction mechanism. These techniques are useful for understanding the principles behind text prediction systems, while modern production systems generally combine much larger datasets with advanced neural language models.
